# B6.5 — Evaluation: all pre-declared analyses (CPU)

Reads the per-image rows written by B6_1–B6_4 and computes **every analysis pre-declared in `results/B6_0_Setup_and_Rules/B6_rule_file.json`** (recorded 2026-10-26, before any test count). No model is run; a **CPU runtime** is enough (≈ 15–30 min, mostly reading ~30,000 row files from Drive; a combined long table is cached, so re-runs take seconds).

| Part | Pre-declared analysis (rule file) |
|---|---|
| 1 | Load all rows → one long table; integrity checks (completeness vs the B6_0 inventory, identical GT across sources, duplicates) |
| 2 | **Single-class** (FSC-147, CARPK): MAE (primary), RMSE, NAE, SRE — all methods; v1.1 headline, v1 derived; ladder usage; paired bootstrap on MAE (primary vs each baseline, primary vs secondary); FSC-147 S1–S6 |
| 3 | **Multi-label** (VOC 2007, OmniCount-191): ours on the full test sets (standard protocol: standard mRMSE primary, mRMSE-nz, MAE_nz); all methods on present classes (full test) and the standard protocol on the fixed subsets (VOC 500, OmniCount Fruits); OmniCount-code metric (not comparable with the paper); VOC non-difficult GT as sensitivity; paired bootstrap on MAE_nz |
| 4 | OmniCount-191 breakdowns: per domain, Fruits subset, without Wild, instance vs area classes |
| 5 | Density analysis: bins 1, 2–5, 6–10, 11–20, 21–50, 51–100, 101–200, >200; counting accuracy, MAE, NAE, samples — all methods, all datasets |
| 6 | Consistency checks against the B6_1–B6_4 progress summaries and `src/evaluation.py` |
| 7 | **Exploratory** (not in the rule file, labelled as such): VOC per-class errors, OmniCount grid64 variant, false positives per absent query |

**Methods.** Ours: GPV_R4+P[MixA] (primary, "Ours A") and GPV_K-n+P[LM0] (secondary, "Ours B"), v1.1 headline / v1 derived on single-class data. References: B0, vanilla SAM (FSC-147, CARPK). Baselines (re-run by us): TFOC (text), OmniCount (declared configuration; primary rule = number of masks; the `count.py` rule — declared as secondary in `B6_4_declarations.json` before its test run — reported alongside as "OmniCount (count.py)").
**Bootstrap:** 10,000 paired resamples of images (seed 0), the same resamples for every comparison within a dataset; Δ = primary − other (negative = primary better), 95% percentile interval, two-sided bootstrap p.

All tables are saved to `results/B6_5_Evaluation/` as CSV, with a headline JSON and the metadata.

In [1]:
# ============================================================
# Step 0.1 — Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.2 — Paths, repo (src.evaluation), the rule file and the B6_0 support files
# ============================================================
import json, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option("display.max_rows", 200); pd.set_option("display.width", 250)
PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"; B60 = RESULTS_ROOT / "B6_0_Setup_and_Rules"
OUT_DIR = RESULTS_ROOT / "B6_5_Evaluation"; OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
FRESH_CLONE = True
if FRESH_CLONE:
    subprocess.run(["rm", "-rf", str(REPO_DIR)])
    subprocess.run(["git", "clone", "-q", "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git", str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR))
import src.evaluation as ev

RULE = json.load(open(B60 / "B6_rule_file.json")); INV = RULE["inventory"]
SUBSETS = json.load(open(B60 / "B6_0_baseline_subsets.json"))
VOC_SUB, FRUITS_SUB = set(SUBSETS["voc_standard_500"]), set(SUBSETS["omnicount_fruits_test"])
STRATA = pd.read_csv(B60 / "B6_0_fsc147_test_gt_counts.csv")                 # dataset_index, density_stratum (val boundaries)
GROUPS = pd.read_csv(B60 / "B6_0_omnicount_class_groups_FINAL.csv")          # domain, class, group (instance / area)
N_BOOT = RULE["statistics"]["paired_bootstrap"]["n"]; SEED = RULE["statistics"]["paired_bootstrap"]["seed"]
BINS = RULE["density_analysis"]["bins"]
PRIMARY = {"single": "Ours A v1.1", "multi": "Ours A"}
SINGLE_METHODS = ["Ours A v1.1", "Ours A v1", "Ours B v1.1", "Ours B v1", "B0", "SAM", "TFOC", "OmniCount", "OmniCount (count.py)"]
MULTI_METHODS = ["Ours A", "Ours B", "TFOC", "OmniCount", "OmniCount (count.py)"]
LABELS = {"Ours A": "GPV_R4+P[MixA] (primary)", "Ours B": "GPV_K-n+P[LM0] (secondary)"}
SAVED = []
def save(df, name, index=False):
    df.to_csv(OUT_DIR / name, index=index); SAVED.append(name)
print("Repo:", REPO_COMMIT[:10], "| rule file recorded", RULE["recorded_utc"], "| bootstrap", N_BOOT, "seed", SEED,
      "| subsets: VOC", len(VOC_SUB), "Fruits", len(FRUITS_SUB), "| class groups", GROUPS["group"].value_counts().to_dict())

Repo: 984bd0b477 | rule file recorded 2026-10-26T13:32:37+00:00 | bootstrap 10000 seed 0 | subsets: VOC 500 Fruits 303 | class groups {'instance': 105, 'area': 15}


In [3]:
# ============================================================
# Step 1.1 — Load every row file (B6_1–B6_4) into one long table: one record per (dataset, image, class, method)
#   pred_present: prediction used in the present-classes protocol (NaN if that class is not evaluated there)
#   pred_std    : prediction under the standard protocol (all vocabulary classes queried; NaN where not run)
#   The table is cached (B6_5_long_table.csv.gz) and rebuilt automatically when any source folder changes.
# ============================================================
R = RESULTS_ROOT
SOURCES = {
    ("fsc147", "ours"): R / "B6_1_Ours_Single" / "fsc147" / "rows_ours", ("carpk", "ours"): R / "B6_1_Ours_Single" / "carpk" / "rows_ours",
    ("fsc147", "b0"): R / "B6_1_Ours_Single" / "fsc147" / "rows_b0", ("carpk", "b0"): R / "B6_1_Ours_Single" / "carpk" / "rows_b0",
    ("omnicount", "ours"): R / "B6_2_Ours_Multi" / "omnicount" / "rows_ours", ("voc", "ours"): R / "B6_2_Ours_Multi" / "voc" / "rows_ours",
    ("fsc147", "sam"): R / "B6_3_Baselines_SAM_TFOC" / "fsc147" / "rows_sam", ("carpk", "sam"): R / "B6_3_Baselines_SAM_TFOC" / "carpk" / "rows_sam",
    **{(d, "tfoc"): R / "B6_3_Baselines_SAM_TFOC" / d / "rows_tfoc" for d in ["fsc147", "carpk", "voc", "omnicount"]},
    **{(d, "omnicount"): R / "B6_4_Baseline_OmniCount" / d / "rows_omnicount" for d in ["fsc147", "carpk", "voc", "omnicount"]},
    ("fsc147", "omnicount_grid64"): R / "B6_4_Baseline_OmniCount" / "fsc147" / "rows_omnicount_grid64",
}
REBUILD = False                      # True forces a rebuild even if the cache matches the sources
LONG_PATH, STAT_PATH = OUT_DIR / "B6_5_long_table.csv.gz", OUT_DIR / "B6_5_long_table_sources.json"

def list_dir(d):
    return sorted(Path(d).glob("*.json")) if Path(d).exists() else []

def read_dir(d):
    files = list_dir(d)
    with ThreadPoolExecutor(32) as ex:
        recs = list(ex.map(lambda p: json.loads(p.read_text()), files))
    out, dup = {}, 0
    for r in recs:                                    # one row per image key (two-session duplicates: first kept)
        if r["key"] in out:
            dup += 1; continue
        out[r["key"]] = r
    return out, dup

def n_errors(r, cls):
    e = r.get("diag", {}).get("errors", {}) or {}
    return sum(1 for k in e if k == cls or k.endswith("|" + cls))

def build_long():
    recs, dups = [], {}
    def add(ds, key, image_id, domain, cls, gt, gt_nd, sub, method, pp, ps, ladder=np.nan, err=0):
        recs.append((ds, key, image_id, domain, cls, gt, gt_nd, sub, method, pp, ps, ladder, err))
    for (ds, fam), d in SOURCES.items():
        t0 = time.time(); rows, dup = read_dir(d); dups[f"{ds}|{fam}"] = dup
        for k, r in rows.items():
            meta = r["meta"]; iid = str(meta.get("image_id", k)); dom = meta.get("domain", ds)
            if ds in ("fsc147", "carpk"):
                c = r["classes"][0]; g = meta["gt"][c]
                if fam == "ours":
                    for s_, lab in [("A", "Ours A"), ("B", "Ours B")]:
                        v = r["counts"][s_][c]; lvl = r["diag"].get(f"{s_}_ladder_level", 0)
                        add(ds, k, iid, dom, c, g, np.nan, False, f"{lab} v1.1", v, np.nan, lvl, n_errors(r, c))
                        add(ds, k, iid, dom, c, g, np.nan, False, f"{lab} v1", 0 if lvl >= 1 else v, np.nan, lvl, n_errors(r, c))
                elif fam in ("b0", "sam", "tfoc"):
                    m = {"b0": "B0", "sam": "SAM", "tfoc": "TFOC"}[fam]
                    add(ds, k, iid, dom, c, g, np.nan, False, m, r["counts"][m][c], np.nan, err=n_errors(r, c))
                else:
                    pre = "OmniCount grid64 (variant)" if fam == "omnicount_grid64" else "OmniCount"
                    add(ds, k, iid, dom, c, g, np.nan, False, pre, r["counts"]["OmniCount"][c], np.nan, err=n_errors(r, c))
                    add(ds, k, iid, dom, c, g, np.nan, False, pre.replace("OmniCount", "OmniCount (count.py)", 1) if fam == "omnicount"
                        else "OmniCount grid64 (count.py, variant)", r["counts"]["OmniCount_countpy"][c], np.nan, err=n_errors(r, c))
                continue
            gt = meta["gt_all"] if ds == "voc" else meta["gt"]
            gt_nd = meta.get("gt_nondifficult", {}) if ds == "voc" else None
            sub = (k in VOC_SUB) if ds == "voc" else (meta["image_id"] in FRUITS_SUB)
            if fam == "ours":                                       # every vocabulary class queried (standard protocol)
                for c in r["classes"]:
                    g = gt.get(c, 0); gn = gt_nd.get(c, 0) if gt_nd is not None else np.nan
                    for s_, lab in [("A", "Ours A"), ("B", "Ours B")]:
                        v = r["counts"][s_][c]
                        add(ds, k, iid, dom, c, g, gn, sub, lab, v if g > 0 else np.nan, v, err=n_errors(r, c))
            elif fam == "tfoc":                                     # present classes everywhere; all classes on the subsets
                for c in r["classes"]:
                    g = gt.get(c, 0); gn = gt_nd.get(c, 0) if gt_nd is not None else np.nan; v = r["counts"]["TFOC"][c]
                    add(ds, k, iid, dom, c, g, gn, sub, "TFOC", v if g > 0 else np.nan, v if sub else np.nan, err=n_errors(r, c))
            else:                                                   # OmniCount: separate present-class and all-class runs
                pres, std = r["classes"], r.get("classes_std") or []
                for c in dict.fromkeys(list(pres) + list(std)):
                    g = gt.get(c, 0); gn = gt_nd.get(c, 0) if gt_nd is not None else np.nan
                    for lab, kp, ks in [("OmniCount", "OmniCount", "OmniCount_std"), ("OmniCount (count.py)", "OmniCount_countpy", "OmniCount_std_countpy")]:
                        pp = r["counts"][kp][c] if (c in pres and g > 0) else np.nan
                        ps = r["counts"][ks][c] if c in std else np.nan
                        add(ds, k, iid, dom, c, g, gn, sub, lab, pp, ps, err=n_errors(r, c))
        print(f"  {ds:9s} {fam:17s} {len(rows):5d} images | duplicates ignored {dup} | {time.time() - t0:.0f}s")
    cols = ["dataset", "key", "image_id", "domain", "cls", "gt_count", "gt_nd", "in_subset", "method", "pred_present", "pred_std", "ladder", "errors"]
    return pd.DataFrame.from_records(recs, columns=cols), dups

src_stats = {f"{ds}|{fam}": len(list_dir(d)) for (ds, fam), d in SOURCES.items()}
cached = LONG_PATH.exists() and STAT_PATH.exists() and json.load(open(STAT_PATH)).get("files") == src_stats
if cached and not REBUILD:
    LONG = pd.read_csv(LONG_PATH, dtype={"key": str, "image_id": str}); DUPS = json.load(open(STAT_PATH))["duplicates"]
    print("Long table loaded from cache:", LONG_PATH.name)
else:
    t0 = time.time(); print("Reading rows from Drive (first run ~10–20 min) ...")
    LONG, DUPS = build_long()
    LONG.to_csv(LONG_PATH, index=False); json.dump({"files": src_stats, "duplicates": DUPS}, open(STAT_PATH, "w"), indent=1)
    print(f"Long table built in {(time.time() - t0) / 60:.1f} min and cached")
print(f"{len(LONG):,} records |", LONG.groupby(["dataset", "method"]).size().unstack(0).fillna(0).astype(int).to_string())

Reading rows from Drive (first run ~10–20 min) ...
  fsc147    ours               1190 images | duplicates ignored 0 | 16s
  carpk     ours                459 images | duplicates ignored 0 | 5s
  fsc147    b0                 1190 images | duplicates ignored 0 | 17s
  carpk     b0                  459 images | duplicates ignored 0 | 6s
  omnicount ours               1957 images | duplicates ignored 0 | 32s
  voc       ours               4952 images | duplicates ignored 0 | 168s
  fsc147    sam                1190 images | duplicates ignored 0 | 15s
  carpk     sam                 459 images | duplicates ignored 0 | 5s
  fsc147    tfoc               1190 images | duplicates ignored 0 | 32s
  carpk     tfoc                459 images | duplicates ignored 0 | 5s
  voc       tfoc               4952 images | duplicates ignored 0 | 163s
  omnicount tfoc               1957 images | duplicates ignored 0 | 28s
  fsc147    omnicount          1190 images | duplicates ignored 1 | 23s
  carpk     omn

In [4]:
# ============================================================
# Step 1.2 — Integrity checks: completeness vs the B6_0 inventory, identical GT across sources, query sets per protocol
# ============================================================
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "ok": bool(ok), "detail": detail})
EXP_IMG = {"fsc147": INV["fsc147_test"]["images"], "carpk": INV["carpk_test"]["images"],
           "voc": INV["voc"]["images"], "omnicount": INV["omnicount_test"]["images"]}
for (ds, m), g in LONG.groupby(["dataset", "method"]):
    check(f"{ds} | {m}: images", g["key"].nunique() == EXP_IMG[ds], f"{g['key'].nunique()}/{EXP_IMG[ds]}")
EXP_PRESENT = {"voc": INV["voc"]["present_queries_all"], "omnicount": INV["omnicount_test"]["present_queries"]}
EXP_STD = {"voc": INV["voc"]["standard_queries"], "omnicount": INV["omnicount_test"]["standard_queries"]}
for ds in ["voc", "omnicount"]:
    L = LONG[LONG.dataset == ds]
    for m in MULTI_METHODS:
        g = L[L.method == m]
        check(f"{ds} | {m}: present-class pairs", g["pred_present"].notna().sum() == EXP_PRESENT[ds],
              f"{g['pred_present'].notna().sum()}/{EXP_PRESENT[ds]}")
    ours_sub = L[(L.method == "Ours A") & L.in_subset]["pred_std"].notna().sum()
    for m in MULTI_METHODS:
        g = L[(L.method == m) & L.in_subset]
        check(f"{ds} | {m}: standard pairs on the subset", g["pred_std"].notna().sum() == ours_sub, f"{g['pred_std'].notna().sum()}/{ours_sub}")
    for m in ["Ours A", "Ours B"]:
        n = L[L.method == m]["pred_std"].notna().sum()
        check(f"{ds} | {m}: standard pairs, full test", n == EXP_STD[ds], f"{n}/{EXP_STD[ds]}")
gt_spread = LONG.groupby(["dataset", "key", "cls"])["gt_count"].agg(lambda s: s.max() - s.min())
check("GT identical across all sources (every image-class)", (gt_spread == 0).all(), f"max spread {gt_spread.max()}")
voc_objects = LONG[(LONG.dataset == "voc") & (LONG.method == "Ours A")]["gt_count"].sum()
check("VOC objects (all) = inventory", voc_objects == INV["voc"]["objects_all"], f"{int(voc_objects)}/{INV['voc']['objects_all']}")
carpk_cars = LONG[(LONG.dataset == "carpk") & (LONG.method == "B0")]["gt_count"].sum()
check("CARPK cars = inventory", carpk_cars == INV["carpk_test"]["cars"], f"{int(carpk_cars)}/{INV['carpk_test']['cars']}")
check("VOC subset size", LONG[(LONG.dataset == "voc") & LONG.in_subset]["key"].nunique() == len(VOC_SUB))
check("OmniCount Fruits subset size", LONG[(LONG.dataset == "omnicount") & LONG.in_subset]["key"].nunique() == len(FRUITS_SUB))
CHECKS = pd.DataFrame(checks); save(CHECKS, "B6_5_integrity_checks.csv")
ERRS = LONG.groupby(["dataset", "method"])["errors"].sum().unstack(0).fillna(0).astype(int)
print("Recorded per-query errors (counted as 0, as pre-declared):"); display(ERRS)
print("Duplicate Drive entries ignored:", {k: v for k, v in DUPS.items() if v})
bad = CHECKS[~CHECKS.ok]
if len(bad):
    display(bad); raise SystemExit(f"INTEGRITY: {len(bad)} check(s) failed — see above")
print(f"INTEGRITY: PASS ({len(CHECKS)} checks)")

Recorded per-query errors (counted as 0, as pre-declared):


dataset,carpk,fsc147,omnicount,voc
method,,,,
B0,0,0,0,0
OmniCount,0,0,0,0
OmniCount (count.py),0,0,0,0
"OmniCount grid64 (count.py, variant)",0,0,0,0
OmniCount grid64 (variant),0,0,0,0
Ours A,0,0,0,0
Ours A v1,0,0,0,0
Ours A v1.1,0,0,0,0
Ours B,0,0,0,0


Duplicate Drive entries ignored: {'fsc147|omnicount': 1, 'voc|omnicount': 1, 'fsc147|omnicount_grid64': 1}
INTEGRITY: PASS (59 checks)


In [5]:
# ============================================================
# Step 1.3 — Metric and bootstrap helpers (definitions as in src/evaluation.py; checked against it in Step 6.1)
# ============================================================
def single_stats(pred, gt):
    pred, gt = np.asarray(pred, float), np.asarray(gt, float); e = pred - gt
    return {"n": len(gt), "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()), "NAE": (np.abs(e) / gt).mean(),
            "SRE": np.sqrt(((e ** 2) / gt).mean())}

def ml_stats(df, pred_col, gt_col="gt_count"):
    """Standard multi-label metrics over the given (image, class) pairs. Classes keyed by (domain, class)."""
    d = df[["image_id", "domain", "cls", gt_col, pred_col]].dropna(subset=[pred_col]).rename(columns={gt_col: "g", pred_col: "p"})
    d["e"] = d["p"] - d["g"]; d["sq"] = d["e"] ** 2; d["ab"] = d["e"].abs(); d["nz"] = d["g"] > 0
    per = d.groupby(["domain", "cls"]).agg(rmse=("sq", lambda s: np.sqrt(s.mean())))
    nzd = d[d.nz]
    per["rmse_nz"] = nzd.groupby(["domain", "cls"])["sq"].apply(lambda s: np.sqrt(s.mean()))
    fp = d.loc[~d.nz, "p"]
    return {"images": d["image_id"].nunique(), "pairs": len(d), "present pairs": int(d.nz.sum()), "classes": len(per),
            "standard mRMSE": per["rmse"].mean(), "mRMSE-nz": per["rmse_nz"].mean(),
            "MAE_nz": nzd["ab"].mean() if len(nzd) else np.nan, "RMSE_nz": np.sqrt(nzd["sq"].mean()) if len(nzd) else np.nan,
            "NAE_nz": (nzd["ab"] / nzd["g"]).mean() if len(nzd) else np.nan,
            "FP units": fp.sum(), "absent pairs": len(fp), "FP per absent query": fp.mean() if len(fp) else np.nan}

def boot_compare(S, N, names, primary, others, dataset, n_boot=None, seed=None):
    """Paired bootstrap over images. S: (n_images, n_methods) per-image sums of |error|; N: (n_images,) pairs per image.
    Statistic = sum S / sum N (MAE for single-class, pooled MAE_nz for multi-label). Same resamples for all methods."""
    n_boot = n_boot or N_BOOT; seed = SEED if seed is None else seed
    rng = np.random.default_rng(seed); n = len(N); reps = []
    for s in range(0, n_boot, 500):
        b = min(500, n_boot - s); idx = rng.integers(0, n, size=(b, n))
        W = np.zeros((b, n)); np.add.at(W, (np.arange(b)[:, None], idx), 1)
        reps.append((W @ S) / (W @ N)[:, None])
    reps = np.vstack(reps); obs = S.sum(0) / N.sum(); ip = names.index(primary); out = []
    for o in others:
        io = names.index(o); dlt = reps[:, ip] - reps[:, io]
        p = min(1.0, 2 * min((dlt >= 0).mean(), (dlt <= 0).mean()))
        out.append({"dataset": dataset, "primary": primary, "other": o, "stat primary": obs[ip], "stat other": obs[io],
                    "delta (primary - other)": obs[ip] - obs[io], "CI 2.5%": np.percentile(dlt, 2.5), "CI 97.5%": np.percentile(dlt, 97.5),
                    "P(primary better)": (dlt < 0).mean(), "p (two-sided)": p, "resamples": n_boot})
    return pd.DataFrame(out)

def bin_label(lo, hi):
    return f"{lo}" if hi == lo else (f">{lo - 1}" if hi is None else f"{lo}-{hi}")
print("Helpers ready")

Helpers ready


In [6]:
# ============================================================
# Step 2.1 — Single-class results (FSC-147, CARPK): MAE (primary), RMSE, NAE, SRE; v1.1 headline, v1 derived; ladder usage
# ============================================================
rows = []
for ds in ["fsc147", "carpk"]:
    L = LONG[LONG.dataset == ds]
    for m in SINGLE_METHODS:
        g = L[L.method == m].sort_values("key")
        rows.append({"dataset": ds, "method": m, **single_stats(g["pred_present"], g["gt_count"]), "errors": int(g["errors"].sum())})
SINGLE = pd.DataFrame(rows); save(SINGLE, "B6_5_single_class_results.csv")
display(SINGLE.pivot(index="method", columns="dataset", values=["MAE", "RMSE", "NAE", "SRE"]).reindex(SINGLE_METHODS).round(2))
PUBLISHED = pd.DataFrame([
    {"dataset": "fsc147", "method": "TFOC (text)", "MAE": 24.79, "RMSE": 137.15, "NAE": 0.37, "SRE": 4.52, "source": "TFOC Table 2"},
    {"dataset": "fsc147", "method": "OmniCount (text)", "MAE": 21.46, "RMSE": 133.28, "NAE": np.nan, "SRE": np.nan,
     "source": "OmniCount Table 2 (its 'NAE' 0.32 / 'SRE' 0.39 use the authors' own definitions)"},
    {"dataset": "fsc147", "method": "SAM", "MAE": 42.48, "RMSE": 137.50, "NAE": 1.14, "SRE": 8.13, "source": "TFOC Tables 1–2"},
    {"dataset": "carpk", "method": "SAM", "MAE": 16.97, "RMSE": 20.57, "NAE": 0.70, "SRE": 5.30, "source": "TFOC Table 1"},
    {"dataset": "carpk", "method": "TFOC (box exemplars; context only)", "MAE": 10.97, "RMSE": 14.24, "NAE": 0.48, "SRE": 3.70,
     "source": "TFOC Table 1 — exemplar-based, not comparable with text prompts"}])
save(PUBLISHED, "B6_5_published_reference.csv")
print("Published figures (reference only):"); display(PUBLISHED)
lad = LONG[LONG.method.isin(["Ours A v1.1", "Ours B v1.1"])].groupby(["dataset", "method"])["ladder"]
LADDER = pd.DataFrame({"ladder fired (L1/L2)": lad.apply(lambda s: int((s >= 1).sum())),
                       "count 0, all levels empty": lad.apply(lambda s: int((s == -1).sum()))}).reset_index()
save(LADDER, "B6_5_ladder_usage.csv"); print("Ladder usage (v1 = v1.1 with these images set to 0):"); display(LADDER)

MAE          RMSE           NAE          SRE       
dataset               carpk fsc147  carpk  fsc147 carpk fsc147 carpk fsc147
method                                                                     
Ours A v1.1            7.27  24.56   9.90  136.59  0.09   0.39  0.93   4.35
Ours A v1              7.27  24.76   9.90  136.91  0.09   0.39  0.93   4.38
Ours B v1.1           15.13  26.87  19.16  139.57  0.15   0.35  1.74   5.24
Ours B v1             15.13  26.88  19.16  139.60  0.15   0.35  1.74   5.25
B0                    30.88  33.06  41.26  144.08  0.61   0.40  4.88   4.91
SAM                   17.27  46.72  20.94  138.70  0.72   1.28  5.48   8.81
TFOC                  37.49  24.70  44.37  137.28  0.40   0.35  4.16   4.43
OmniCount             49.67  29.34  55.14  137.53  0.54   0.45  5.22   4.89
OmniCount (count.py)  77.74  44.63  83.11  148.22  0.76   0.55  7.68   5.82

Published figures (reference only):


,dataset,method,MAE,RMSE,NAE,SRE,source
0,fsc147,TFOC (text),24.79,137.15,0.37,4.52,TFOC Table 2
1,fsc147,OmniCount (text),21.46,133.28,NaN,NaN,OmniCount Table 2 (its 'NAE' 0.32 / 'SRE' 0.39...
2,fsc147,SAM,42.48,137.50,1.14,8.13,TFOC Tables 1–2
3,carpk,SAM,16.97,20.57,0.70,5.30,TFOC Table 1
4,carpk,TFOC (box exemplars; context only),10.97,14.24,0.48,3.70,"TFOC Table 1 — exemplar-based, not comparable ..."


Ladder usage (v1 = v1.1 with these images set to 0):


,dataset,method,ladder fired (L1/L2),"count 0, all levels empty"
0,carpk,Ours A v1.1,0,0
1,carpk,Ours B v1.1,0,0
2,fsc147,Ours A v1.1,3,0
3,fsc147,Ours B v1.1,3,0


In [7]:
# ============================================================
# Step 2.2 — Paired bootstrap on MAE (single-class): primary vs each baseline / reference, primary vs secondary
# ============================================================
boots = []
OTHERS_SINGLE = ["Ours B v1.1", "B0", "SAM", "TFOC", "OmniCount", "OmniCount (count.py)"]
for ds in ["fsc147", "carpk"]:
    L = LONG[LONG.dataset == ds]
    P = L[L.method.isin(SINGLE_METHODS)].pivot(index="key", columns="method", values="pred_present")
    G = L[L.method == "B0"].set_index("key")["gt_count"].reindex(P.index)
    names = [PRIMARY["single"]] + OTHERS_SINGLE
    S = np.abs(P[names].values - G.values[:, None]); N = np.ones(len(G))
    boots.append(boot_compare(S, N, names, PRIMARY["single"], OTHERS_SINGLE, ds))
BOOT_SINGLE = pd.concat(boots, ignore_index=True); save(BOOT_SINGLE, "B6_5_bootstrap_single_class.csv")
display(BOOT_SINGLE.round(4))

,dataset,primary,other,stat primary,stat other,delta (primary - other),CI 2.5%,CI 97.5%,P(primary better),p (two-sided),resamples
0,fsc147,Ours A v1.1,Ours B v1.1,24.563,26.8731,-2.3101,-4.0328,-0.7588,0.9982,0.0036,10000
1,fsc147,Ours A v1.1,B0,24.563,33.0588,-8.4958,-10.3337,-6.7294,1.0000,0.0000,10000
2,fsc147,Ours A v1.1,SAM,24.563,46.7244,-22.1613,-24.2555,-19.9823,1.0000,0.0000,10000
3,fsc147,Ours A v1.1,TFOC,24.563,24.6983,-0.1353,-1.8933,1.5714,0.5593,0.8814,10000
4,fsc147,Ours A v1.1,OmniCount,24.563,29.3395,-4.7765,-6.4085,-3.1705,1.0000,0.0000,10000
5,fsc147,Ours A v1.1,OmniCount (count.py),24.563,44.6252,-20.0622,-22.2849,-17.8933,1.0000,0.0000,10000
6,carpk,Ours A v1.1,Ours B v1.1,7.268,15.1285,-7.8606,-8.5251,-7.2113,1.0000,0.0000,10000
7,carpk,Ours A v1.1,B0,7.268,30.8802,-23.6122,-25.7560,-21.5185,1.0000,0.0000,10000
8,carpk,Ours A v1.1,SAM,7.268,17.2658,-9.9978,-11.2833,-8.6950,1.0000,0.0000,10000
9,carpk,Ours A v1.1,TFOC,7.268,37.4880,-30.2200,-32.3509,-28.1002,1.0000,0.0000,10000


In [8]:
# ============================================================
# Step 2.3 — FSC-147 S1–S6 (boundaries from FSC-147 val, B6_0): MAE and NAE per stratum, all methods
# ============================================================
st = STRATA.assign(key=STRATA["dataset_index"].map(lambda i: f"{int(i):05d}"))[["key", "density_stratum"]]
F = LONG[(LONG.dataset == "fsc147") & LONG.method.isin(SINGLE_METHODS)].merge(st, on="key", how="left")
assert F["density_stratum"].notna().all(), "stratum missing for some FSC-147 images"
rows = []
for (s, m), g in F.groupby(["density_stratum", "method"]):
    rows.append({"stratum": s, "method": m, **single_stats(g["pred_present"], g["gt_count"])})
STR = pd.DataFrame(rows); save(STR, "B6_5_fsc147_strata.csv")
print("MAE by stratum:"); display(STR.pivot(index="method", columns="stratum", values="MAE").reindex(SINGLE_METHODS).round(2))
print("Images per stratum:", STR[STR.method == PRIMARY["single"]].set_index("stratum")["n"].to_dict())

MAE by stratum:


stratum,S1_0_20,S2_20_40,S3_40_60,S4_60_80,S5_80_95,S6_95_100
method,,,,,,
Ours A v1.1,8.29,8.26,10.44,14.18,30.06,256.91
Ours A v1,8.29,8.26,10.44,14.18,30.21,261.02
Ours B v1.1,6.97,6.28,7.49,14.18,36.06,313.00
Ours B v1,6.97,6.28,7.49,14.18,36.09,313.13
B0,4.89,7.01,9.49,17.34,54.75,343.53
SAM,27.80,36.39,36.76,41.48,50.41,216.57
TFOC,6.15,6.89,8.83,14.56,35.94,248.38
OmniCount,8.21,8.97,11.18,18.23,45.58,264.21
OmniCount (count.py),4.51,7.88,14.18,31.81,82.42,371.40


Images per stratum: {'S1_0_20': 142, 'S2_20_40': 186, 'S3_40_60': 277, 'S4_60_80': 291, 'S5_80_95': 247, 'S6_95_100': 47}


In [9]:
# ============================================================
# Step 3.1 — Multi-label results (VOC 2007, OmniCount-191)
#   (a) ours, full test, standard protocol (all classes queried): standard mRMSE (primary), mRMSE-nz, MAE_nz
#   (b) all methods, full test, present classes: MAE_nz, mRMSE-nz (standard mRMSE undefined without absent queries)
#   (c) all methods, standard protocol on the fixed subsets (VOC 500, OmniCount Fruits)
#   (d) OmniCount-code metric (present classes; port of the authors' metric — NOT comparable with the paper's split/numbers)
#   (e) VOC sensitivity: non-difficult GT
# ============================================================
rows = []
for ds in ["voc", "omnicount"]:
    L = LONG[LONG.dataset == ds]
    for m in ["Ours A", "Ours B"]:
        rows.append({"dataset": ds, "protocol": "standard, full test", "gt_conv": "all", "method": m, **ml_stats(L[L.method == m], "pred_std")})
    for m in MULTI_METHODS:
        rows.append({"dataset": ds, "protocol": "present classes, full test", "gt_conv": "all", "method": m, **ml_stats(L[L.method == m], "pred_present")})
    for m in MULTI_METHODS:
        g = L[(L.method == m) & L.in_subset]
        rows.append({"dataset": ds, "protocol": "standard, subset", "gt_conv": "all", "method": m, **ml_stats(g, "pred_std")})
    if ds == "voc":                                             # sensitivity: non-difficult objects only
        for m in ["Ours A", "Ours B"]:
            rows.append({"dataset": ds, "protocol": "standard, full test", "gt_conv": "non-difficult", "method": m,
                         **ml_stats(L[L.method == m], "pred_std", "gt_nd")})
        for m in MULTI_METHODS:
            g = L[(L.method == m) & (L["gt_nd"] > 0)].copy(); g["pp_nd"] = g["pred_present"]
            rows.append({"dataset": ds, "protocol": "present classes, full test", "gt_conv": "non-difficult", "method": m, **ml_stats(g, "pp_nd", "gt_nd")})
            g = L[(L.method == m) & L.in_subset]
            rows.append({"dataset": ds, "protocol": "standard, subset", "gt_conv": "non-difficult", "method": m, **ml_stats(g, "pred_std", "gt_nd")})
MULTI = pd.DataFrame(rows)
MULTI.loc[MULTI.protocol.str.startswith("present"), "standard mRMSE"] = np.nan
save(MULTI, "B6_5_multi_label_results.csv")
cols = ["standard mRMSE", "mRMSE-nz", "MAE_nz", "FP per absent query"]
for proto in ["standard, full test", "present classes, full test", "standard, subset"]:
    print(f"\n{proto} (GT: all objects)")
    t = MULTI[(MULTI.protocol == proto) & (MULTI["gt_conv"] == "all")]
    pv = t.pivot(index="method", columns="dataset", values=[c for c in cols if t[c].notna().any()])
    display(pv.reindex([m for m in MULTI_METHODS if m in pv.index]).round(3))
OCODE = []
for ds in ["voc", "omnicount"]:
    L = LONG[(LONG.dataset == ds) & LONG.pred_present.notna()]
    for m in MULTI_METHODS:
        g = L[L.method == m]
        res = [{"image_id": iid, "domain": gg["domain"].iloc[0], "categories": {c: {"count": n} for c, n in zip(gg["cls"], gg["gt_count"])},
                "pred_counts": dict(zip(gg["cls"], gg["pred_present"]))} for iid, gg in g.groupby("image_id")]
        _, om_ = ev.omnicount_reference_by_domain(res)
        OCODE.append({"dataset": ds, "method": m, **om_})
OCODE = pd.DataFrame(OCODE); save(OCODE, "B6_5_omnicount_code_metric.csv")
print("\nOmniCount-code metric (present classes; authors' normalisation — not comparable with the paper):"); display(OCODE.round(4))


standard, full test (GT: all objects)


standard mRMSE         mRMSE-nz            MAE_nz        FP per absent query       
dataset      omnicount    voc omnicount     voc omnicount    voc           omnicount    voc
method                                                                                     
Ours A           2.239  2.234     7.308   8.198     2.849  4.912               0.361  0.158
Ours B           2.679  4.599     8.787  15.851     3.359  6.936               0.347  0.219


present classes, full test (GT: all objects)


mRMSE-nz            MAE_nz       
dataset              omnicount     voc omnicount    voc
method                                                 
Ours A                   7.308   8.198     2.849  4.912
Ours B                   8.787  15.851     3.359  6.936
TFOC                    15.650  15.134    10.620  8.647
OmniCount               14.180  15.542     8.273  8.717
OmniCount (count.py)     4.448   4.006     2.773  2.042


standard, subset (GT: all objects)


standard mRMSE          mRMSE-nz            MAE_nz        FP per absent query        
dataset                   omnicount     voc omnicount     voc omnicount    voc           omnicount     voc
method                                                                                                    
Ours A                        0.784   2.109     0.878   7.573     0.583  4.438               0.234   0.159
Ours B                        1.157   4.316     1.453  13.249     0.589  6.503               0.213   0.253
TFOC                          4.567  12.678     4.100  12.796     3.824  7.394               4.758   6.704
OmniCount                     6.587  31.324     5.120  12.979     4.036  7.958               7.161  23.604
OmniCount (count.py)          2.484   7.687     1.681   3.190     1.152  1.819               2.669   5.494


OmniCount-code metric (present classes; authors' normalisation — not comparable with the paper):


,dataset,method,pooled_m_rmse,pooled_m_rmse_nz,mean_over_domains_m_rmse,mean_over_domains_m_rmse_nz
0,voc,Ours A,0.0080,0.0331,0.0080,0.0331
1,voc,Ours B,0.0154,0.0640,0.0154,0.0640
2,voc,TFOC,0.0150,0.0611,0.0150,0.0611
3,voc,OmniCount,0.0160,0.0628,0.0160,0.0628
4,voc,OmniCount (count.py),0.0042,0.0162,0.0042,0.0162
5,omnicount,Ours A,0.0280,0.3586,0.5749,1.1227
6,omnicount,Ours B,0.0335,0.4329,0.6788,1.3632
7,omnicount,TFOC,0.0734,0.7496,1.7592,3.2719
8,omnicount,OmniCount,0.0668,0.6996,1.2046,2.4372
9,omnicount,OmniCount (count.py),0.0214,0.2197,0.4871,0.8379


In [10]:
# ============================================================
# Step 3.2 — Paired bootstrap on MAE_nz (multi-label, present classes, full test): primary vs each baseline, primary vs secondary
# ============================================================
boots = []
OTHERS_MULTI = ["Ours B", "TFOC", "OmniCount", "OmniCount (count.py)"]
for ds in ["voc", "omnicount"]:
    L = LONG[(LONG.dataset == ds) & LONG.pred_present.notna() & LONG.method.isin(MULTI_METHODS)].copy()
    L["ae"] = (L["pred_present"] - L["gt_count"]).abs()
    S = L.pivot_table(index="key", columns="method", values="ae", aggfunc="sum")
    Nm = L.pivot_table(index="key", columns="method", values="ae", aggfunc="size")
    assert (Nm.nunique(axis=1) == 1).all(), "methods differ in present pairs per image"
    names = [PRIMARY["multi"]] + OTHERS_MULTI
    boots.append(boot_compare(S[names].values, Nm[names[0]].values.astype(float), names, PRIMARY["multi"], OTHERS_MULTI, ds))
BOOT_MULTI = pd.concat(boots, ignore_index=True); save(BOOT_MULTI, "B6_5_bootstrap_multi_label.csv")
display(BOOT_MULTI.round(4))

,dataset,primary,other,stat primary,stat other,delta (primary - other),CI 2.5%,CI 97.5%,P(primary better),p (two-sided),resamples
0,voc,Ours A,Ours B,4.9121,6.9359,-2.0238,-2.2890,-1.7734,1.0000,0.0000,10000
1,voc,Ours A,TFOC,4.9121,8.6474,-3.7353,-4.0078,-3.4669,1.0000,0.0000,10000
2,voc,Ours A,OmniCount,4.9121,8.7170,-3.8049,-4.1038,-3.5297,1.0000,0.0000,10000
3,voc,Ours A,OmniCount (count.py),4.9121,2.0418,2.8703,2.7286,3.0102,0.0000,0.0000,10000
4,omnicount,Ours A,Ours B,2.8490,3.3588,-0.5099,-0.7074,-0.3407,1.0000,0.0000,10000
5,omnicount,Ours A,TFOC,2.8490,10.6204,-7.7714,-8.2964,-7.2648,1.0000,0.0000,10000
6,omnicount,Ours A,OmniCount,2.8490,8.2728,-5.4238,-5.8946,-4.9555,1.0000,0.0000,10000
7,omnicount,Ours A,OmniCount (count.py),2.8490,2.7730,0.0759,-0.1370,0.2973,0.2455,0.4922,10000


In [11]:
# ============================================================
# Step 4.1 — OmniCount-191 breakdowns: per domain, Fruits subset, without Wild, instance vs area classes
# ============================================================
O = LONG[LONG.dataset == "omnicount"].merge(GROUPS[["domain", "class", "group"]].rename(columns={"class": "cls"}), on=["domain", "cls"], how="left")
assert O["group"].notna().all(), "class group missing for some OmniCount classes"
rows = []
def add_rows(view, label, df):
    for m in MULTI_METHODS:
        g = df[df.method == m]
        r = {"breakdown": view, "level": label, "method": m, **{f"present: {k}": v for k, v in ml_stats(g, "pred_present").items()
                                                                 if k in ("images", "present pairs", "MAE_nz", "mRMSE-nz")}}
        if m in ("Ours A", "Ours B"):
            r.update({f"standard (ours): {k}": v for k, v in ml_stats(g, "pred_std").items() if k in ("standard mRMSE", "mRMSE-nz", "MAE_nz")})
        rows.append(r)
for d in sorted(O["domain"].unique()):
    add_rows("domain", d, O[O.domain == d])
add_rows("Fruits subset", "Fruits", O[O.in_subset])
add_rows("without Wild", "all domains except Wild", O[O.domain != "Wild"])
for gname in ["instance", "area"]:
    add_rows("class group", gname, O[O.group == gname])
BRK = pd.DataFrame(rows); save(BRK, "B6_5_omnicount_breakdowns.csv")
for view in ["domain", "Fruits subset", "without Wild", "class group"]:
    t = BRK[BRK.breakdown == view]
    print(f"\n{view}: present classes MAE_nz")
    display(t.pivot(index="method", columns="level", values="present: MAE_nz").reindex(MULTI_METHODS).round(3))
print("\nOurs, standard protocol (all classes queried) — standard mRMSE:")
display(BRK[BRK.method.isin(["Ours A", "Ours B"])].pivot(index=["breakdown", "level"], columns="method", values="standard (ours): standard mRMSE").round(3))


domain: present classes MAE_nz


level,Birds,Fruits,Pets,Satellite,Supermarket,Urban,Wild
method,,,,,,,
Ours A,5.552,0.583,1.962,14.277,11.870,2.244,1.652
Ours B,5.345,0.589,2.736,15.669,11.382,3.212,2.154
TFOC,5.897,3.824,10.528,30.469,18.283,12.219,13.772
OmniCount,2.897,4.033,7.925,17.700,19.881,9.180,5.097
OmniCount (count.py),4.345,1.150,1.849,4.431,7.812,2.991,2.468



Fruits subset: present classes MAE_nz


level,Fruits
method,
Ours A,0.583
Ours B,0.589
TFOC,3.824
OmniCount,4.033
OmniCount (count.py),1.150



without Wild: present classes MAE_nz


level,all domains except Wild
method,
Ours A,2.941
Ours B,3.451
TFOC,10.379
OmniCount,8.516
OmniCount (count.py),2.796



class group: present classes MAE_nz


level,area,instance
method,,
Ours A,15.677,2.502
Ours B,17.657,2.972
TFOC,25.778,10.210
OmniCount,15.253,8.084
OmniCount (count.py),3.747,2.747



Ours, standard protocol (all classes queried) — standard mRMSE:


method                                 Ours A  Ours B
breakdown     level                                  
Fruits subset Fruits                    0.784   1.157
class group   area                      4.733   7.570
              instance                  1.883   1.981
domain        Birds                     6.137   5.823
              Fruits                    0.784   1.157
              Pets                      0.911   1.301
              Satellite                 4.092   5.913
              Supermarket               2.039   1.806
              Urban                     1.350   2.333
              Wild                      2.814   3.362
without Wild  all domains except Wild   2.219   2.656

In [12]:
# ============================================================
# Step 5.1 — Density analysis (unit: one GT count — an image for single-class, an image-class pair for multi-label,
#   present classes). Bins 1, 2–5, 6–10, 11–20, 21–50, 51–100, 101–200, >200. Counting accuracy = mean(max(0, 1 − |e|/gt)) × 100.
# ============================================================
rows = []
for ds, methods in [("fsc147", SINGLE_METHODS), ("carpk", SINGLE_METHODS), ("voc", MULTI_METHODS), ("omnicount", MULTI_METHODS)]:
    L = LONG[(LONG.dataset == ds) & LONG.method.isin(methods) & LONG.pred_present.notna()]
    for lo, hi in BINS:
        sel = L[(L["gt_count"] >= lo) & ((L["gt_count"] <= hi) if hi is not None else True)]
        for m in methods:
            g = sel[sel.method == m]
            if not len(g):
                rows.append({"dataset": ds, "bin": bin_label(lo, hi), "method": m, "samples": 0}); continue
            e = (g["pred_present"] - g["gt_count"]).abs()
            rows.append({"dataset": ds, "bin": bin_label(lo, hi), "method": m, "samples": len(g),
                         "counting accuracy (%)": 100 * np.maximum(0, 1 - e / g["gt_count"]).mean(), "MAE": e.mean(), "NAE": (e / g["gt_count"]).mean()})
DENS = pd.DataFrame(rows); save(DENS, "B6_5_density_analysis.csv")
order = [bin_label(lo, hi) for lo, hi in BINS]
for ds in ["fsc147", "carpk", "voc", "omnicount"]:
    t = DENS[(DENS.dataset == ds) & (DENS.samples > 0)]
    print(f"\n{ds}: counting accuracy (%) by GT bin | samples per bin:",
          t[t.method == t.method.iloc[0]].set_index("bin")["samples"].reindex([b for b in order if b in set(t.bin)]).to_dict())
    display(t.pivot(index="method", columns="bin", values="counting accuracy (%)").reindex(columns=[b for b in order if b in set(t.bin)]).round(1))


fsc147: counting accuracy (%) by GT bin | samples per bin: {'6-10': 60, '11-20': 268, '21-50': 413, '51-100': 254, '101-200': 139, '>200': 56}


bin,6-10,11-20,21-50,51-100,101-200,>200
method,,,,,,
B0,52.5,59.5,67.1,66.1,52.5,37.9
OmniCount,47.8,57.1,65.2,68.0,61.9,55.5
OmniCount (count.py),60.7,53.0,48.1,38.5,30.5,24.7
Ours A v1,43.5,57.2,67.5,76.9,74.7,64.4
Ours A v1.1,43.5,57.2,67.5,76.9,74.9,65.7
Ours B v1,55.6,64.9,74.3,77.7,69.2,47.9
Ours B v1.1,55.6,64.9,74.3,77.7,69.3,47.9
SAM,4.3,7.7,21.6,46.6,64.9,69.9
TFOC,56.0,63.1,71.0,74.2,69.6,62.7



carpk: counting accuracy (%) by GT bin | samples per bin: {'2-5': 10, '6-10': 10, '11-20': 6, '21-50': 21, '51-100': 111, '101-200': 301}


bin,2-5,6-10,11-20,21-50,51-100,101-200
method,,,,,,
B0,0.0,0.0,22.5,71.9,70.9,72.9
OmniCount,18.5,42.6,39.2,49.5,45.5,52.8
OmniCount (count.py),34.5,18.9,18.8,23.4,22.7,25.1
Ours A v1,64.7,63.9,89.1,87.6,92.0,93.5
Ours A v1.1,64.7,63.9,89.1,87.6,92.0,93.5
Ours B v1,72.2,75.6,83.0,77.6,84.2,86.3
Ours B v1.1,72.2,75.6,83.0,77.6,84.2,86.3
SAM,0.0,0.0,0.0,30.6,86.9,86.3
TFOC,25.0,22.5,12.6,48.1,56.8,66.3



voc: counting accuracy (%) by GT bin | samples per bin: {'1': 4904, '2-5': 2317, '6-10': 331, '11-20': 71, '21-50': 9}


bin,1,2-5,6-10,11-20,21-50
method,,,,,
OmniCount,10.2,18.8,29.4,36.0,54.6
OmniCount (count.py),38.0,47.6,53.2,43.9,24.0
Ours A,18.9,27.5,48.8,58.0,63.7
Ours B,26.6,34.0,49.1,56.6,64.7
TFOC,11.5,23.3,38.1,38.7,43.0



omnicount: counting accuracy (%) by GT bin | samples per bin: {'1': 1961, '2-5': 1347, '6-10': 329, '11-20': 55, '21-50': 42, '51-100': 20}


bin,1,2-5,6-10,11-20,21-50,51-100
method,,,,,,
OmniCount,6.1,41.5,47.9,35.5,22.5,47.4
OmniCount (count.py),39.6,41.1,31.5,41.2,59.4,24.5
Ours A,38.5,44.9,51.0,40.7,15.4,6.2
Ours B,42.2,43.9,43.7,37.3,15.4,7.6
TFOC,4.6,16.4,23.7,28.8,18.8,46.6


In [13]:
# ============================================================
# Step 6.1 — Consistency checks: (a) helpers vs src/evaluation.py on real data; (b) every number already printed by
#   the B6_1–B6_4 progress summaries is reproduced here (same rows, same definitions)
# ============================================================
cons = []
for ds in ["voc", "omnicount"]:                                  # (a) ml_stats vs ev.evaluate_multi_label_counts
    for m, col, sub in [("Ours A", "pred_std", False), ("TFOC", "pred_present", False), ("OmniCount", "pred_std", True)]:
        g = LONG[(LONG.dataset == ds) & (LONG.method == m) & LONG[col].notna()]
        if sub:
            g = g[g.in_subset]
        res = [{"image_id": iid, "domain": gg["domain"].iloc[0],
                "categories": {c: {"count": n} for c, n in zip(gg["cls"], gg["gt_count"]) if n > 0},
                "pred_counts": dict(zip(gg["cls"], gg[col])), "query_classes": list(gg["cls"])} for iid, gg in g.groupby("image_id")]
        _, _, ref = ev.evaluate_multi_label_counts(ev.build_multi_label_pairs(res)); mine = ml_stats(g, col)
        for a, b in [("mRMSE", "standard mRMSE"), ("mRMSE_nz", "mRMSE-nz"), ("MAE_nz", "MAE_nz"), ("false_positive_units", "FP units")]:
            cons.append({"check": f"src.evaluation | {ds} | {m} | {col} | {b}", "expected": ref[a], "got": mine[b]})
PROTO = {"single-class": "single", "present classes (full test)": "present classes, full test", "standard (subset)": "standard, subset"}
def lookup_multi(ds, proto, gt, m):
    t = MULTI[(MULTI.dataset == ds) & (MULTI.protocol == proto) & (MULTI["gt_conv"] == gt) & (MULTI.method == m)]
    return t.iloc[0] if len(t) else None
progress = {"B6_1": RESULTS_ROOT / "B6_1_Ours_Single" / "B6_1_progress_summary.csv",
            "B6_2": RESULTS_ROOT / "B6_2_Ours_Multi" / "B6_2_progress_summary.csv",
            "B6_3": RESULTS_ROOT / "B6_3_Baselines_SAM_TFOC" / "B6_3_progress_summary.csv",
            "B6_4": RESULTS_ROOT / "B6_4_Baseline_OmniCount" / "B6_4_progress_summary_A.csv"}
NAME = {"GPV_R4+P[MixA]": "Ours A", "GPV_K-n+P[LM0]": "Ours B", "OmniCount_countpy": "OmniCount (count.py)"}
def canon(s):
    for a, b in NAME.items():
        s = s.replace(a, b)
    return s
for nb, path in progress.items():
    if not path.exists():
        print("not found (skipped):", path); continue
    P = pd.read_csv(path)
    for _, r in P.iterrows():
        if nb == "B6_2":
            ds, m, gt, proto = r["dataset"], canon(r["system"]), ("non-difficult" if r["gt"] == "gt_nondifficult" else "all"), "standard, full test"
        else:
            ds, m = r["dataset"], canon(r["method"]); proto = PROTO.get(r.get("protocol", "single-class"), "single"); gt = "all"
        if ds in ("fsc147", "carpk"):
            mine = SINGLE[(SINGLE.dataset == ds) & (SINGLE.method == m)]
            if not len(mine):
                continue
            for k in ["MAE", "RMSE", "NAE", "SRE"]:
                if k in P.columns and pd.notna(r[k]):
                    cons.append({"check": f"{nb} | {ds} | {m} | {k}", "expected": r[k], "got": mine.iloc[0][k]})
        else:
            mine = lookup_multi(ds, proto, gt, m)
            if mine is None:
                continue
            for k in ["standard mRMSE", "mRMSE-nz", "MAE_nz", "RMSE_nz", "NAE_nz", "FP units"]:
                if k in P.columns and pd.notna(r[k]) and pd.notna(mine[k]):
                    cons.append({"check": f"{nb} | {ds} | {proto} | {gt} | {m} | {k}", "expected": r[k], "got": mine[k]})
            if "OmniCount m_rmse_nz" in P.columns and pd.notna(r.get("OmniCount m_rmse_nz")):
                oc = OCODE[(OCODE.dataset == ds) & (OCODE.method == m)]
                cons.append({"check": f"{nb} | {ds} | {m} | OmniCount-code m_rmse_nz", "expected": r["OmniCount m_rmse_nz"], "got": oc.iloc[0]["pooled_m_rmse_nz"]})
CONS = pd.DataFrame(cons)
CONS["ok"] = np.isclose(CONS["expected"].astype(float), CONS["got"].astype(float), rtol=1e-6, atol=1e-9)
save(CONS, "B6_5_consistency_checks.csv")
print(f"{len(CONS)} comparisons | by source:", CONS["check"].str.split(" | ").str[0].value_counts().to_dict())
if not CONS.ok.all():
    display(CONS[~CONS.ok]); raise SystemExit("CONSISTENCY: mismatches — see above")
print("CONSISTENCY: PASS — every progress-summary number reproduced; helpers identical to src/evaluation.py")

176 comparisons | by source: {'B6_4': 44, 'B6_1': 40, 'B6_2': 38, 'B6_3': 30, 'src.evaluation': 24}
CONSISTENCY: PASS — every progress-summary number reproduced; helpers identical to src/evaluation.py


In [14]:
# ============================================================
# Step 7.1 — EXPLORATORY (not in the rule file): VOC per-class errors; OmniCount grid64 variant (FSC-147);
#   false positives per absent query on the subsets
# ============================================================
V = LONG[LONG.dataset == "voc"]
rows = []
for c, g in V.groupby("cls"):
    r = {"class": c, "present pairs": int(g[g.method == "Ours A"]["pred_present"].notna().sum())}
    for m in MULTI_METHODS:
        gp = g[(g.method == m) & g.pred_present.notna()]
        r[f"MAE_nz {m}"] = (gp["pred_present"] - gp["gt_count"]).abs().mean()
        r[f"bias {m}"] = (gp["pred_present"] - gp["gt_count"]).mean()
    ga = g[(g.method == "Ours A") & (g["gt_count"] == 0)]
    r["Ours A: mean prediction on absent (full test)"] = ga["pred_std"].mean()
    rows.append(r)
VPC = pd.DataFrame(rows).sort_values("MAE_nz Ours A", ascending=False); save(VPC, "B6_5_exploratory_voc_per_class.csv")
print("VOC per class (present classes; positive bias = overcounting):")
display(VPC[["class", "present pairs"] + [f"MAE_nz {m}" for m in MULTI_METHODS] + ["bias Ours A", "Ours A: mean prediction on absent (full test)"]].round(2))
G64 = LONG[(LONG.dataset == "fsc147") & LONG.method.str.contains("grid64")]
if len(G64):
    gv = pd.DataFrame([{"method": m, **single_stats(g["pred_present"], g["gt_count"])} for m, g in G64.groupby("method")])
    save(gv, "B6_5_exploratory_omnicount_grid64.csv"); print("\nOmniCount grid64 variant on FSC-147 (pre-declared variant; failed the gate):"); display(gv.round(3))
fp = MULTI[(MULTI.protocol == "standard, subset") & (MULTI["gt_conv"] == "all")][["dataset", "method", "absent pairs", "FP units", "FP per absent query"]]
save(fp, "B6_5_exploratory_fp_per_absent_query.csv"); print("\nFalse positives on absent queries (standard protocol, subsets):"); display(fp.round(3))

VOC per class (present classes; positive bias = overcounting):


,class,present pairs,MAE_nz Ours A,MAE_nz Ours B,MAE_nz TFOC,MAE_nz OmniCount,MAE_nz OmniCount (count.py),bias Ours A,Ours A: mean prediction on absent (full test)
10,diningtable,247,16.46,15.55,12.75,19.02,4.34,16.40,0.43
18,train,259,14.16,25.21,25.21,20.90,4.31,14.16,0.04
5,bus,183,11.97,10.97,20.03,18.28,3.72,11.95,0.11
15,pottedplant,254,10.31,17.80,8.42,10.31,2.40,10.24,1.18
3,boat,176,7.39,10.28,6.13,9.31,2.28,7.03,0.04
6,car,775,6.36,9.24,11.97,12.01,2.53,6.22,0.10
0,aeroplane,205,6.02,8.31,7.87,12.05,2.51,5.98,0.01
13,motorbike,233,5.32,8.70,23.89,20.36,4.15,5.09,0.02
17,sofa,355,4.74,4.77,9.83,7.69,1.93,4.57,0.25
1,bicycle,250,4.08,11.35,11.09,7.40,1.68,3.82,0.02



OmniCount grid64 variant on FSC-147 (pre-declared variant; failed the gate):


,method,n,MAE,RMSE,NAE,SRE
0,"OmniCount grid64 (count.py, variant)",1190,43.580,147.559,0.538,5.725
1,OmniCount grid64 (variant),1190,30.276,137.343,0.499,5.155



False positives on absent queries (standard protocol, subsets):


,dataset,method,absent pairs,FP units,FP per absent query
7,voc,Ours A,9236,1469.0,0.159
8,voc,Ours B,9236,2339.0,0.253
9,voc,TFOC,9236,61919.0,6.704
10,voc,OmniCount,9236,218008.0,23.604
11,voc,OmniCount (count.py),9236,50738.0,5.494
31,omnicount,Ours A,1198,280.0,0.234
32,omnicount,Ours B,1198,255.0,0.213
33,omnicount,TFOC,1198,5700.0,4.758
34,omnicount,OmniCount,1198,8579.0,7.161
35,omnicount,OmniCount (count.py),1198,3197.0,2.669


In [15]:
# ============================================================
# Step 8.1 — Headline numbers, metadata, list of saved tables
# ============================================================
def pick(df, **kw):
    t = df.copy()
    for k, v in kw.items():
        t = t[t[k] == v]
    return t.iloc[0]
HEAD = {"single_class": {ds: {m: {k: float(pick(SINGLE, dataset=ds, method=m)[k]) for k in ["MAE", "RMSE", "NAE", "SRE"]}
                              for m in SINGLE_METHODS} for ds in ["fsc147", "carpk"]},
        "multi_label": {ds: {proto: {m: {k: (None if pd.isna(v) else float(v)) for k, v in pick(MULTI, dataset=ds, protocol=proto, gt_conv="all", method=m)[
                              ["standard mRMSE", "mRMSE-nz", "MAE_nz", "FP per absent query"]].items()}
                              for m in (["Ours A", "Ours B"] if proto == "standard, full test" else MULTI_METHODS)}
                              for proto in ["standard, full test", "present classes, full test", "standard, subset"]} for ds in ["voc", "omnicount"]},
        "bootstrap": pd.concat([BOOT_SINGLE, BOOT_MULTI]).round(6).to_dict(orient="records")}
json.dump(HEAD, open(OUT_DIR / "B6_5_headline.json", "w"), indent=1); SAVED.append("B6_5_headline.json")
json.dump({"notebook": "B6_5_Evaluation", "repo_commit": REPO_COMMIT, "rule_file_recorded_utc": RULE["recorded_utc"],
           "bootstrap": {"n": N_BOOT, "seed": SEED, "unit": "image", "statistic": "MAE (single) / pooled MAE_nz (multi)",
                         "p_value": "two-sided percentile bootstrap", "interval": "95% percentile"},
           "sources": src_stats, "duplicates_ignored": DUPS, "integrity_checks": int(len(CHECKS)), "consistency_checks": int(len(CONS)),
           "methods": {"single": SINGLE_METHODS, "multi": MULTI_METHODS, "labels": LABELS},
           "omnicount_note": "declared configuration (B6_4_declarations.json); gate overridden after review (FSC-147 MAE 29.34 vs 21.46); "
                             "count.py rule declared as secondary before the test run",
           "saved": SAVED, "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")},
          open(OUT_DIR / "B6_5_metadata.json", "w"), indent=2)
print("Saved to", OUT_DIR); print("\n".join(sorted(SAVED + ["B6_5_metadata.json", LONG_PATH.name])))
print("\nB6_5: COMPLETE")

Saved to /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B6_5_Evaluation
B6_5_bootstrap_multi_label.csv
B6_5_bootstrap_single_class.csv
B6_5_consistency_checks.csv
B6_5_density_analysis.csv
B6_5_exploratory_fp_per_absent_query.csv
B6_5_exploratory_omnicount_grid64.csv
B6_5_exploratory_voc_per_class.csv
B6_5_fsc147_strata.csv
B6_5_headline.json
B6_5_integrity_checks.csv
B6_5_ladder_usage.csv
B6_5_long_table.csv.gz
B6_5_metadata.json
B6_5_multi_label_results.csv
B6_5_omnicount_breakdowns.csv
B6_5_omnicount_code_metric.csv
B6_5_published_reference.csv
B6_5_single_class_results.csv

B6_5: COMPLETE
